# Load and explore the data

Load Online Retail II, save a fast Parquet copy, and investigate its data quality issues. The findings here define the rules in the mapping file.

## 1. Imports



In [2]:
import pandas as pd
pd.set_option("display.max_rows",100)

## 2. Load both Excel sheets and combine them

Read both sheets with `sheet_name=None`, then combine with `pd.concat`.

In [4]:
sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)
df = pd.concat(sheets.values(),ignore_index= True)
print("Sheets loaded:", list(sheets.keys()))

Sheets loaded: ['Year 2009-2010', 'Year 2010-2011']


## 3. Save a Parquet copy

Future notebooks load this in seconds.

In [6]:
for col in ["Invoice","StockCode","Description"]:
    df[col] = df[col].astype("string")
df.to_parquet("../data/raw/online_retail_II.parquet", index=False)
print("Saved")

Saved


## 4. First inspection

Shape, column names, first rows, data types, missing values.

In [7]:
print("Rows, columns:", df.shape)
print()
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isna().sum())
df.head()

Rows, columns: (1067371, 8)

Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Missing values per column:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 5. Duplicate rows



In [8]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 34335


## 6. Cancelled invoices

Invoices starting with "C".

In [9]:
cancelled = df["Invoice"].str.startswith("C")
print("Cancelled rows:", cancelled.sum())
print("Cancelled invoices:", df.loc[cancelled, "Invoice"].nunique())

Cancelled rows: 19494
Cancelled invoices: 8292


## 7. Zero or negative quantities and prices



In [10]:
print("Rows with quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Rows with price <= 0:", (df["Price"] <= 0).sum())
print("Negative quantity rows that are cancellations:", ((df["Quantity"] < 0) & cancelled).sum())

Rows with quantity <= 0: 22950
Rows with price <= 0: 6207
Negative quantity rows that are cancellations: 19493


## 8. Missing Customer ID

Count and percentage of rows.

In [11]:
missing = df["Customer ID"].isna().sum()
print("Rows without Customer ID:", missing)
print("Percentage:", round(missing / len(df) * 100, 1), "%")

Rows without Customer ID: 243007
Percentage: 22.8 %


## 9. Non-product stock codes

Stock codes that don't start with a digit, with their descriptions.

In [12]:
non_product = ~df["StockCode"].str.match(r"^\d")
codes = (df.loc[non_product, ["StockCode", "Description"]]
           .drop_duplicates("StockCode")
           .sort_values("StockCode"))
print("Non-product codes found:", len(codes))
codes

Non-product codes found: 62


,StockCode,Description
70975,ADJUST,Adjustment by john on 26/01/2010 16
249672,ADJUST2,Adjustment by Peter on Jun 25 2010
440688,AMAZONFEE,AMAZON FEE
179403,B,Adjust bad debt
18410,BANK CHARGES,Bank Charges
9292,C2,CARRIAGE
302041,C3,<NA>
842969,CRUK,CRUK Commission
735,D,Discount
20249,DCGS0003,BOXED GLASS ASHTRAY


## 10. Summary of findings

Write your findings here in plain English. They become the mapping rules in Step 3.

In [ ]:
- 1,067,371 rows covering Dec 2009 – Dec 2011.
- 34,335 exact duplicate rows (3.2%), mainly from the overlap between the two sheets → remove.
- 8,292 cancelled invoices (19,494 rows, invoice starts with "C") → exclude from purchases, but count per customer as a possible dissatisfaction signal.
- Almost all negative quantities are cancellations; the remaining ~3,450 are stock adjustments → exclude.
- 6,207 rows with price ≤ 0 → exclude.
- 22.8% of rows have no Customer ID (likely guest checkouts) → exclude from customer-level modelling; noted as a limitation.
- A "starts with a digit" rule would wrongly remove real products (DCGS…, PADS, SP1002), so non-product codes are defined as a reviewed list: service charges, accounting entries, test products and gift vouchers.